# Automated Interpretability

# Automated Interpretability

## What This Is
Manual neuron labeling doesn't scale to models with billions of neurons. Automated interpretability uses LLMs to generate natural language descriptions of what neurons and features do.

**Protocol (Bills et al., 2023 - GPT-4 for neuron labeling)**:
1. For each neuron, collect top-k activating text examples (max-activating examples)
2. Feed examples to an LLM with a prompt: 'These text snippets cause this neuron to activate. Write a description of what they have in common.'
3. Evaluate the description by checking if it can predict neuron activations on held-out examples

**Evaluation**: Use the LLM description to score new texts, then compare predicted activations with actual activations. High correlation = good description.

In [ ]:
import numpy as np
from typing import List, Dict, Tuple

np.random.seed(42)

# Automated interpretability pipeline
# Simulate neuron activation patterns and LLM-generated descriptions

# Token vocabulary for demo
TOKENS = [
    'dog', 'cat', 'bird', 'fish', 'lion', 'tiger', 'bear',  # animals
    'run', 'jump', 'fly', 'swim', 'walk',  # motion verbs
    'red', 'blue', 'green', 'yellow',  # colors
    'big', 'small', 'fast', 'slow',  # adjectives
    'the', 'a', 'is', 'in', 'on',  # function words
]
TOK2IDX = {t: i for i, t in enumerate(TOKENS)}

# Simulate neuron activations
# Neuron 0: activates for animals (tokens 0-6)
# Neuron 1: activates for motion verbs (tokens 7-11)
# Neuron 2: polysemantic: colors + adjectives

def neuron_activation(token: str, neuron_idx: int) -> float:
    idx = TOK2IDX.get(token, -1)
    if neuron_idx == 0:  # animal neuron
        return 1.0 if idx < 7 else 0.1 * np.random.random()
    elif neuron_idx == 1:  # motion verb neuron
        return 1.0 if 7 <= idx < 12 else 0.1 * np.random.random()
    elif neuron_idx == 2:  # polysemantic: colors OR adjectives
        return 1.0 if 12 <= idx < 20 else 0.1 * np.random.random()
    return np.random.random() * 0.2

# Find max-activating examples for each neuron
def get_max_activating(neuron_idx: int, top_k: int = 5) -> List[str]:
    acts = [(tok, neuron_activation(tok, neuron_idx)) for tok in TOKENS]
    return [t for t, a in sorted(acts, key=lambda x: x[1], reverse=True)[:top_k]]

# Simulate LLM description generation
def simulate_llm_description(max_activating: List[str]) -> str:
    """
    In production: call LLM API to generate description.
    Here we use simple heuristics.
    """
    animal_words = {'dog', 'cat', 'bird', 'fish', 'lion', 'tiger', 'bear'}
    motion_words = {'run', 'jump', 'fly', 'swim', 'walk'}
    color_words = {'red', 'blue', 'green', 'yellow'}
    adj_words = {'big', 'small', 'fast', 'slow'}
    
    max_set = set(max_activating)
    if max_set & animal_words: return 'animal names and living creatures'
    if max_set & motion_words: return 'verbs describing physical movement'
    if max_set & (color_words | adj_words): return 'descriptive adjectives (colors and size/speed)'
    return 'miscellaneous tokens'

def evaluate_description(description: str, neuron_idx: int, eval_tokens: List[str]) -> float:
    """
    Score correlation between description-predicted and actual activations.
    Simplified: check if high-activation tokens match the description.
    """
    # Predicted: which tokens match the description
    animal_kw = ['animal', 'creature', 'living']
    motion_kw = ['movement', 'motion', 'physical']
    adj_kw = ['descriptive', 'adjective', 'color', 'size', 'speed']
    
    def desc_score(tok, desc):
        idx = TOK2IDX.get(tok, -1)
        if any(k in desc for k in animal_kw): return 1.0 if idx < 7 else 0.0
        if any(k in desc for k in motion_kw): return 1.0 if 7 <= idx < 12 else 0.0
        if any(k in desc for k in adj_kw): return 1.0 if 12 <= idx < 20 else 0.0
        return 0.5
    
    pred = np.array([desc_score(t, description) for t in eval_tokens])
    actual = np.array([neuron_activation(t, neuron_idx) for t in eval_tokens])
    if pred.std() < 1e-8: return 0.0
    return float(np.corrcoef(pred, actual)[0, 1])

print('Automated Interpretability Pipeline:')
print('=' * 55)
for n in range(4):
    max_acts = get_max_activating(n)
    desc = simulate_llm_description(max_acts)
    eval_corr = evaluate_description(desc, n, TOKENS)
    print(f'Neuron {n}:')
    print(f'  Max activating: {max_acts}')
    print(f'  LLM description: "{desc}"')
    print(f'  Eval correlation: {eval_corr:.3f}')
    print()
